# 04 - Raven Discharge Ensemble Forecasts

## Learning goals of this module
- Learn how to do a basic analysis, comparing observed discharge with forecast discharge, produced by Raven and forced by GEPS, GEFS and IFS.
- Learn about ensemble forecast verification.
- Learn how to calculate and visualize simple error metrics, such as the __rank_histogram__ and the __continuous_ranked_probability_score__.

## Assumptions
- We assume you are familiar with the concept of a __probabilistic__ forecast. 
- We assume you are familiar with the concept of a __hydrological__ model.

### Reference to ensemble forecast products

- [Global Ensemble Prediction System (GEPS)](https://open.canada.ca/data/en/dataset/6d9dd2f8-202e-58cb-a110-e2168832aacb)

- [Global Ensemble Forecast System (GEFS)](https://www.ncei.noaa.gov/products/weather-climate-models/global-ensemble-forecast)

- [Integrated Forecasting System (IFS)](https://www.ecmwf.int/en/forecasts/documentation-and-support/changes-ecmwf-model)

### Reference to Raven
[Raven Hydrological Model](https://raven.uwaterloo.ca/)


## Run imports and set-up logging

In [1]:
import logging
import sys
import warnings
from pathlib import Path

from dotenv import load_dotenv

from veriflow import run_pipeline
from veriflow.constants import VERSION

# add project root (parent of notebook folder) to path
sys.path.append(str(Path("..").resolve()))


from tree_plots import (
    plot_score_vs_lead_time,
    forecast_timeseries_plot,
    get_pair_dataset,
    rank_histogram_3d_plot,
    rank_histogram_plot,
    scatter_plot,
)

# Reload automatically
%load_ext autoreload
%autoreload 2


warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

load_dotenv(dotenv_path="tutorial.env", override=True)

base_config = Path("config")
base_config.exists()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()],
)
logging.info(f"Running Veriflow version {VERSION}")

Exception in thread Thread-5 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\beunk\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\beunk\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\beunk\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "C:\Users\beunk\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 28536: character maps to <undefined>
2026-09-17 14:44:24,323 - INFO - Run

## Inspecting the _veriflow_ pipeline configuration
1. Open the config file in the "config" directory. The name of the file is identical to the name of the notebook.
2. Inspect each of the sections to gain an understanding of what this configuration is about.

## Running the _veriflow_ pipeline

In [2]:
dt = run_pipeline((base_config / "04_raven_elbow_discharge_ensemble.yaml", "yaml"))

2026-09-17 14:44:24,399 - INFO - Successfully initialized the configuration. 
	 verification_period_start = 2025-05-20 00:00:00 
	 verification_period_end = 2026-06-01 00:00:00
2026-09-17 14:44:24,402 - INFO - Starting dataset fetch (source_id=observed) from FewsWebservice.
2026-09-17 14:44:24,734 - INFO - Download successful from URL: https://veriflow-open.fews.deltares.nl/FewsWebServices/rest/fewspiservice/v1/timeseries?locationIds=05BJ010&parameterIds=QR.obs&moduleInstanceIds=ImportWSC&startTime=2025-05-21T00%3A00%3A00Z&endTime=2026-06-07T00%3A00%3A00Z&timeSeriesType=EXTERNAL_HISTORICAL&documentFormat=PI_NETCDF
2026-09-17 14:44:24,736 - INFO - Starting dataset fetch (source_id=observed) from FewsNetCDF.
2026-09-17 14:44:25,625 - INFO - Dataset (source_id=observed) successfully loaded and validated.
2026-09-17 14:44:25,625 - INFO - Starting dataset fetch (source_id=simulated_GEPS) from FewsWebservice.
2026-09-17 14:44:26,100 - INFO - Download successful from URL: https://veriflow-ope

In [3]:
dt

<xarray.DataTree 'veriflow-datatree'>
Group: /
├── Group: /input_data
│   ├── Group: /input_data/observed
│   │       Dimensions:       (station: 1, time: 383)
│   │       Coordinates:
│   │         * station       (station) <U7 28B '05BJ010'
│   │           station_name  (station) |S255 255B b'ELBOW RIVER AT SARCEE BRIDGE'
│   │           lat           (station) float64 8B 50.99
│   │           lon           (station) float64 8B -114.2
│   │           y             (station) float64 8B 50.99
│   │           x             (station) float64 8B -114.2
│   │           z             (station) float64 8B nan
│   │         * time          (time) datetime64[ns] 3kB 2025-05-21 2025-05-22 ... 2026-06-07
│   │       Data variables:
│   │           discharge     (station, time) float32 2kB 8.36 7.97 8.08 ... 59.8 48.7 39.8
│   │       Attributes: (12/14)
│   │           originalParameterId:  QR.obs
│   │           Conventions:          CF-1.6
│   │           coordinate_system:    WGS 1984
│   │           featureType:          timeSeries
│   │           time_coverage_start:  2025-05-21T00:00:00+0000
│   │           time_coverage_end:    2026-06-07T00:00:00+0000
│   │           ...                   ...
│   │           geospatial_lat_min:   50.9935
│   │           geospatial_lat_max:   50.9935
│   │           data_type:            observed_historical
│   │           source_id:            observed
│   │           spatial_type:         point
│   │           crs:                  EPSG:4326
│   ├── Group: /input_data/simulated_GEPS
│   │       Dimensions:                  (station: 1, forecast_reference_time: 18,
│   │                                     lead_time: 6, realization: 21)
│   │       Coordinates:
│   │         * station                  (station) <U7 28B '05BJ010'
│   │           station_name             (station) |S255 255B b'ELBOW RIVER AT SARCEE BRI...
│   │           lat                      (station) float64 8B 50.99
│   │           lon                      (station) float64 8B -114.2
│   │           y                        (station) float64 8B 50.99
│   │           x                        (station) float64 8B -114.2
│   │           z                        (station) float64 8B nan
│   │         * forecast_reference_time  (forecast_reference_time) datetime64[ns] 144B 20...
│   │         * lead_time                (lead_time) timedelta64[ns] 48B 1 days ... 6 days
│   │           time                     (forecast_reference_time, lead_time) datetime64[ns] 864B ...
│   │         * realization              (realization) int32 84B 0 1 2 3 4 ... 17 18 19 20
│   │       Data variables:
│   │           discharge                (station, forecast_reference_time, lead_time, realization) float32 9kB ...
│   │       Attributes: (12/14)
│   │           originalParameterId:  QR.sim
│   │           Conventions:          CF-1.6
│   │           coordinate_system:    WGS 1984
│   │           featureType:          timeSeries
│   │           time_coverage_start:  2026-05-15T07:00:00+0000
│   │           time_coverage_end:    2026-05-22T07:00:00+0000
│   │           ...                   ...
│   │           geospatial_lat_min:   50.9935
│   │           geospatial_lat_max:   50.9935
│   │           data_type:            simulated_forecast_ensemble
│   │           source_id:            simulated_GEPS
│   │           spatial_type:         point
│   │           crs:                  EPSG:4326
│   ├── Group: /input_data/simulated_GEFS
│   │       Dimensions:                  (station: 1, forecast_reference_time: 18,
│   │                                     lead_time: 6, realization: 31)
│   │       Coordinates:
│   │         * station                  (station) <U7 28B '05BJ010'
│   │           station_name             (station) |S255 255B b'ELBOW RIVER AT SARCEE BRI...
│   │           lat                      (station) float64 8B 50.99
│   │           lon                      (station) float64 8B -114.2
│   │           y                        (st

## Evaluating the results in the _veriflow_ output `DataTree`

Verification metrics and results can contain a level of abstraction. Although these abstractions can reveal important information about forecast quality, a basic "eyeball verification" is often the best and intuitive way to start your verification exercise. You'll likely find strengths and weaknesses in your forecasts early on, without directly diving into levels of abstraction. In addition, a solid visual inspection may help you later on in understanding or explaining the more abstract results.

### 1 - Visual inspection of observed and forecast data
A good starting point for "eyeball" verification is simple: just looking at your observations and forecasts in a visual way. Use the interactive elements in the plots below to zoom, pan and compare the results of our 3 NWP products.

In [4]:
stations = get_pair_dataset(dt, dt.veriflow.verification_pairs[0]).coords["station"].values
lead_times = get_pair_dataset(dt, dt.veriflow.verification_pairs[0]).coords["lead_time"].values

In [5]:
forecast_timeseries_plot(dt)

### 2 - Visual inspection with a scatter plot per lead time
Another great tool for "eyeball" verification is the scatter plot. The scatter plot is relatively easy to understand, but is slightly more abstract than the visualization above. 

For all forecasts in our output `DataTree`, we collect all realizations of the ensemble at a specific lead time. You can use the  `lead_times` variable  (a list of `np.timedelta64` instances) to slice the data at a specific lead time. For that given slice, we can make the scatter plot.


In [6]:
scatter_plot(dt, lead_time=lead_times[1])

### 3 - Looking into the Continuous Ranked Probability Score

Next, we'll look into the results of the continuous ranked probability score. Before you continue to the next section, we'll look into the documentation and do a tutorial on the CRPS for ensemble forecasts. The _veriflow_ package relies on _scores_ (developed by the Bureau of Meteorology, Australia) for computation of various scores. 

1. Read the [API documentation](https://scores.readthedocs.io/en/stable/api.html#scores.probability.crps_for_ensemble) the __crps_for_ensemble__ function, which is used under the hood in _veriflow_. 
2. Run through the [scores tutorial](https://scores.readthedocs.io/en/stable/tutorials/CRPS_for_Ensembles.html) on the __crps_for_ensemble__ function. You can run it in Binder (link on top of the page), or view the static view.


- Q1: what attribute(s) of forecast quality can be measured by the CRPS?

<details>
<summary>Show suggested answers for Q1</summary>
The CRPS captures multiple attributes: accuracy, reliability and sharpness. Can you reason why?

</details>

- Q2: what is the best possible CRPS score?

<details>
<summary>Show suggested answers for Q2</summary>
The best possible outcome of CRPS is 0. In this case the forecast has maximum sharpness: all ensemble members exactly predict the observed outcome.
</details>


- Q3: computing the CRPS over just one realization (i.e. a deterministic forecast) yields the exact same result as computing the ... for a deterministic score?

<details>
<summary>Show suggested answers for Q3</summary>
The absolute error. The CRPS is a probabilistic generalization of the absolute error. When taking the mean of the CRPS over all forecasts, it is equal to the mean absolute error when the number of realizations is 1.
</details>

In [7]:
plot_score_vs_lead_time(dt)

### 4 - Looking into the Rank Histogram

Next, we'll look into the results of the rank histogram. Before you continue to the next section, we'll look into the documentation and do a tutorial on the rank histogram for ensemble forecasts. The _veriflow_ package relies on _scores_ (developed by the Bureau of Meteorology, Australia) for computation of various scores. 

1. Read the [API documentation](https://scores.readthedocs.io/en/stable/api.html#scores.probability.rank_histogram) the __rank_histogram__ function, which is used under the hood in _veriflow_. 
2. Run through the [scores tutorial](https://scores.readthedocs.io/en/stable/tutorials/Rank_Histogram.html) on the __rank_histogram__ function. You can run it in Binder (link on top of the page), or view the static view.

- Q1: what attribute(s) of forecast quality can be measured by the Rank Histogram?

<details>
<summary>Show suggested answers for Q1</summary>
The rank histogram primarily measures reliability, although bias in forecasts will show up in the rank histogram as well.
</details>

- Q2: what does a perfect Rank Histogram look like?

<details>
<summary>Show suggested answers for Q2</summary>
The rank histogram shows a uniform (flat) distribution, indicating forecast and observed frequencies are equal.
</details>

- Q3: does a perfectly uniform/flat rank histogram always correspond to a good forecast? Can you come up with a hypothetical forecast that has a perfect rank histogram, but is still bad?

<details>
<summary>Show suggested answers for Q3</summary>
No, although a perfect rank histogram indicates statistical reliability, the forecast may still have poor resolution (the ability to discriminate between situations that have different observed outcomes).

Imagine an ensemble forecast that ignores current conditions and simply samples from the historical distribution of streamflow for that day of year.

For example, every day in June the ensemble consists of random draws from the last 30 years of June observations.

This forecast can also produce a nearly uniform rank histogram because the observations come from the same climatological distribution. However, it has zero ability to distinguish today's conditions from any other June day.

</details>


In [8]:
rank_histogram_plot(dt, station=stations[0], lead_time=lead_times[4])

### 5 - Looking into the Rank Histogram in 3D (advanced)
This might seem relatively abstract or advanced, but the 3D rank histogram shows the rank histogram for all lead times. In the current notebook, the number of samples is too low to make a proper analysis of the results, but with many samples this can be an interesting and useful visualization.


In [9]:
rank_histogram_3d_plot(dt, station=stations[0])